Module 6 Homework

In this homework we'll put what we learned about Spark in practice.


For this homework we will be using the Yellow 2025-11 data from the official website:

In [24]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-04 22:40:19--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.117, 52.85.39.153, 52.85.39.65, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.117|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M   354MB/s    in 0.2s    

2026-03-04 22:40:19 (354 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [25]:
!ls


sample_data			 yellow_tripdata_2025-11.parquet.1
taxi_zone_lookup.csv		 yellow_tripdata_2025-11_repartitioned
yellow_tripdata_2025-11.parquet


Question 1:

Install Spark and PySpark

Install Spark

Run PySpark

Create a local spark session

Execute spark.version.

What's the output?

In [26]:
import pyspark
from pyspark.sql import SparkSession

In [27]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homewwork') \
    .getOrCreate()

In [28]:
spark.version

'4.0.2'

Question 2:
Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [29]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [30]:
df_repart = df.repartition(4)

In [31]:
print("Particiones:", df_repart.rdd.getNumPartitions())

Particiones: 4


In [32]:
df_repart.write \
    .mode("overwrite") \
    .parquet("yellow_tripdata_2025-11_repartitioned")

In [33]:
!ls -lh yellow_tripdata_2025-11_repartitioned

total 102M
-rw-r--r-- 1 root root 26M Mar  4 22:40 part-00000-2eefdfe3-0bb0-48ca-82ea-425cb7b0b0b8-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  4 22:40 part-00001-2eefdfe3-0bb0-48ca-82ea-425cb7b0b0b8-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  4 22:41 part-00002-2eefdfe3-0bb0-48ca-82ea-425cb7b0b0b8-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  4 22:41 part-00003-2eefdfe3-0bb0-48ca-82ea-425cb7b0b0b8-c000.snappy.parquet
-rw-r--r-- 1 root root   0 Mar  4 22:41 _SUCCESS


Question 3:
Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [34]:
from pyspark.sql.functions import to_date, col

df_15 = df.filter(
    to_date(col("tpep_pickup_datetime")) == "2025-11-15"
)

df_15.count()

162604

In [35]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

Question 4:

Longest trip

What is the length of the longest trip in the dataset in hours?

In [36]:
from pyspark.sql.functions import col, unix_timestamp, max

df_with_duration = df.withColumn(
    "trip_duration_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) -
     unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

df_with_duration.select(max("trip_duration_hours")).show()

+------------------------+
|max(trip_duration_hours)|
+------------------------+
|       90.64666666666666|
+------------------------+



Question 5:

User Interface

Spark's User Interface which shows the application's dashboard runs on which local port? 4040

Question 6:

Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [37]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-04 22:41:05--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.65, 52.85.39.153, 52.85.39.97, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-04 22:41:05 (236 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [38]:
!ls

sample_data		yellow_tripdata_2025-11.parquet
taxi_zone_lookup.csv	yellow_tripdata_2025-11.parquet.1
taxi_zone_lookup.csv.1	yellow_tripdata_2025-11_repartitioned


In [39]:
df_zonas = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [40]:
df.createOrReplaceTempView("yellow")
df_zonas.createOrReplaceTempView("zones")

In [43]:
result = spark.sql("""
SELECT
    z.Zone,
    COUNT(*) AS trip_count
FROM yellow y
JOIN zones z
    ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trip_count ASC
LIMIT 5
""")

result.show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Arden Heights                                |1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Governor's Island/Ellis Island/Liberty Island|1         |
|Port Richmond                                |3         |
|Rikers Island                                |4         |
+---------------------------------------------+----------+

